In [1]:
import ee
import geemap
import xarray as xr

In [2]:
ee.Authenticate()
ee.Initialize(
    project='ee-gabriel-495521',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

In [3]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
roi = map.draw_last_feature.geometry()
roi

ee.Geometry({
  "functionInvocationValue": {
    "functionName": "Feature.geometry",
    "arguments": {
      "feature": {
        "functionInvocationValue": {
          "functionName": "Feature",
          "arguments": {
            "geometry": {
              "functionInvocationValue": {
                "functionName": "GeometryConstructors.Polygon",
                "arguments": {
                  "coordinates": {
                    "constantValue": [
                      [
                        [
                          43.725586,
                          35.603719
                        ],
                        [
                          43.725586,
                          41.804078
                        ],
                        [
                          50.493164,
                          41.804078
                        ],
                        [
                          50.493164,
                          35.603719
                        ],
                        [
                          43.725586,
                          35.603719
                        ]
                      ]
                    ]
                  },
                  "geodesic": {
                    "constantValue": false
                  }
                }
              }
            }
          }
        }
      }
    }
  }
})

In [6]:
temp = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .filterDate('2020', '2021')
    .select('LST_1KM')
)
temp

In [7]:
ndvi = (
    ee.ImageCollection("NASA/VIIRS/002/VNP13A1")
    .filterDate('2020', '2021')
    .select('NDVI')
)

ndvi

In [ ]:
ds_temp = xr.open_dataset(
    temp,
    engine='ee',
    crs = 'EPSG:4326',
    geometry = roi,
    scale = 0.01
)

ds_temp

TypeError: EarthEngineBackendEntrypoint.open_dataset() got an unexpected keyword argument 'geometry'

In [ ]:
ds_ndvi = xr.open_dataset(
    ndvi,
    engine='ee',
    crs='EPSG:4326',
    geometry=roi,
    scale=0.005
)

ndvi

In [ ]:
ds_temp=ds_temp.sortby('time') * 1
ds_ndvi=ds_ndvi.sortby('time') * 1

In [ ]:
# Converter para o escala mensal
ds_temp_monthly = ds_temp.resample(time='ME').mean('time')
ds_ndvi_monthly = ds_ndvi.resample(time='ME').median('time')

NameError: name 'ds_temp' is not defined

In [ ]:
ds_temp_monthly

NameError: name 'ds_temp_monthly' is not defined

In [ ]:
ds_ndvi_monthly

NameError: name 'ds_ndvi_monthly' is not defined

In [ ]:
# Match datasets espacialmente
ds_ndvi_monthly_regrid = ds_ndvi_monthly.interp_like(ds_temp_monthly)
ds_ndvi_monthly_regrid

NameError: name 'ds_ndvi_monthly' is not defined

In [ ]:
ds_temp_monthly


NameError: name 'ds_temp_monthly' is not defined

In [ ]:
# Empacotar os dois datasets
ds_merge = xr.merge(ds_temp_monthly, ds_ndvi_monthly_regrid)
ds_merge

NameError: name 'ds_temp_monthly' is not defined

In [ ]:
# Exportar netcdf